### Read Metadata


In [0]:
metadata_df = spark.table(
    "cicd_dev.bronze.ingestion_metadata"
)

display(metadata_df)

### Generic Bronze Loader

In [0]:
metadata = metadata_df.collect()

for row in metadata:

    source_table = row["source_table"]
    bronze_table = row["bronze_table"]

    print(f"Processing {source_table}")

### Read Source and Write Bronze

In [0]:
from pyspark.sql.functions import current_timestamp

metadata = metadata_df.collect()

for row in metadata:

    source_table = row["source_table"]
    bronze_table = row["bronze_table"]

    source_path = f"cicd_dev.default.{source_table}"
    target_path = f"cicd_dev.bronze.{bronze_table}"

    try:
        df = spark.table(source_path)
        df.write.format("delta").mode("overwrite").saveAsTable(target_path)
        record_count = spark.table(target_path).count()

        audit_data = [
            (
                bronze_table,
                record_count,
                "SUCCESS"
            )
        ]

        audit_df = spark.createDataFrame(
            audit_data,
            ["table_name","record_count","status"]
        ).select(
            "*", current_timestamp().alias("load_time")
        )

        audit_df.write.mode("append").saveAsTable(
            "cicd_dev.bronze.ingestion_audit"
        )

    except Exception as e:
        audit_data = [
            (
                bronze_table,
                0,
                f"FAILED : {str(e)}"
            )
        ]

        audit_df = spark.createDataFrame(
            audit_data,
            ["table_name","record_count","status"]
        ).select(
            "*", current_timestamp().alias("load_time")
        )

        audit_df.write.mode("append").saveAsTable(
            "cicd_dev.bronze.ingestion_audit"
        )

In [0]:
%sql
SELECT *
FROM cicd_dev.bronze.ingestion_audit
ORDER BY load_time DESC;